# 05 · Mejora medida del retrieval

**Objetivo:** Mejorar la búsqueda midiendo recall@k contra el ancla de texto tras cada cambio.

**Requisitos:** R08, R11 ([01](../docs/01_requisitos_y_contratos.md)) · **Guía:** [11](../docs/11_skill_mejora_retrieval.md) · teoría: [04](../docs/04_teoria_rag_retrieval.md)

**Entradas:** `golden/golden_propio.jsonl`, índice FAISS · **Salidas:** `resultados/retrieval/`

**Independiente:** se ejecuta solo, sin ejecutar antes otros notebooks: lee `data/` y lo guardado en `resultados/`. Con `EJECUTAR = False` no llama a la API.

**Modelo:** usa `config.MODELO_ID`, por defecto `openrouter:inclusionai/ling-3.0-flash-fin:free`, el principal elegido en el banco del notebook 02. Reinicia el kernel tras cambiar la configuración. El cambio de modelo genera otro experimento y otra clave de caché; los resultados anteriores de Gemini se conservan y no se reutilizan.

In [1]:
# Arranque: usa el kernel env_agentes y las rutas del paquete.
import sys
import pathlib
import json
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ / "src"))
from agente10k import config, datos, evaluacion, retrieval, herramientas

print("Python:", sys.executable)


Python: c:\Users\Fernando Dapena T\Desktop\Agentes\env_agentes\Scripts\python.exe


In [2]:
# Solo True permite nuevas llamadas de reescritura a OpenRouter.
# Con False, los pasos locales funcionan y la reescritura solo usa la caché.
EJECUTAR = True
REUTILIZAR_RANKINGS = True  # False repite búsquedas locales; conserva la caché LLM.
RUTA_PREDICCIONES_AGENTE = config.RESULTADOS / "final" / "predicciones.jsonl"

# La huella del experimento (preparar_retrieval) incluye el modelo, asi que cada modelo escribe
# en su propio directorio y no se mezclan escaleras. Lo unico que hay que vigilar es el gasto:
# con un modelo de pago, EJECUTAR=True cobra las reescrituras que no esten ya en cache.
if not config.MODELO_ID.startswith("openrouter:"):
    raise ValueError("El notebook 05 requiere un modelo de OpenRouter. "
                     "Revisa AGENTE10K_MODELO y reinicia el kernel.")
if EJECUTAR and not config.MODELO_ID.endswith(":free"):
    print(f"AVISO: {config.MODELO_ID} no es gratuito; las reescrituras sin cache se cobraran.")

# No se carga ni se muestra la clave aquí: reescribir() la leerá de .env
# únicamente si EJECUTAR=True y falta una entrada válida de la caché.
golden = evaluacion.cargar_golden(config.GOLDEN / "golden_propio.jsonl")
con_ancla = [p for p in golden if p.get("ancla_texto")]
DIRECTORIO = evaluacion.preparar_retrieval(golden)
print(f"Golden: {len(golden)} preguntas; retrieval: {len(con_ancla)} con ancla.")
print("Anclas no indexables: 0. Configuración fijada antes de medir.")
print("Resultados de este experimento:", DIRECTORIO)
print("Modelo de reescritura:", config.MODELO_ID, "| temperatura:", config.TEMPERATURA)
print("API habilitada:", EJECUTAR)
print("n_cand:", config.RETRIEVAL_N_CAND, "| constante RRF:", config.RETRIEVAL_K_RRF)

AVISO: openrouter:google/gemini-3.8-flash no es gratuito; las reescrituras sin cache se cobraran.
Golden: 20 preguntas; retrieval: 13 con ancla.
Anclas no indexables: 0. Configuración fijada antes de medir.
Resultados de este experimento: C:\Users\Fernando Dapena T\Desktop\Agentes\10k-financial-agent\resultados\retrieval\7d0a40294471
Modelo de reescritura: openrouter:google/gemini-3.8-flash | temperatura: 0
API habilitada: True
n_cand: 20 | constante RRF: 60


### Qué mide esta comparación

El Paso 0 busca cada `pregunta` de `golden/golden_propio.jsonl`: son las 13 preguntas con `ancla_texto`, no una única consulta escrita en esta celda. La tabla del Paso 0 muestra las preguntas completas. Se envían en español y sin filtros a un modelo de embeddings en inglés; aquí no interviene el agente financiero.

Los pasos 1 y 2 usan filtros oráculo del golden para aislar el efecto del filtro y de BM25 + RRF. El Paso 3 usa el LLM y conserva empresa, ejercicios y sección cuando aparecen explícitamente en la pregunta, sin consultar las respuestas del golden. En comparativas conserva ambos ejercicios.

Un acierto exige encontrar el ancla en la empresa, ejercicio y sección esperados. Las reescrituras con errores no se reutilizan como respuestas válidas: con `EJECUTAR=True` se reintentan, con un máximo de dos intentos por pregunta. Un error de cuota o autenticación detiene la tanda y conserva las respuestas correctas en caché. Los cambios de código o protocolo generan otro directorio de resultados.


## 1. Paso 0: búsqueda densa y recall@k de partida

Guía: [11 §3–§4](../docs/11_skill_mejora_retrieval.md)

In [3]:
# Comprueba con dos preguntas reales que el nuevo camino denso conserva
# exactamente el ranking del baseline, incluidos los empates de FAISS.
for pregunta in con_ancla[:2]:
    original = retrieval.buscar_denso(pregunta["pregunta"], k=20)
    nuevo = retrieval.buscar_hibrido([pregunta["pregunta"]], k=20, usar_bm25=False)
    assert [d["chunk_id"] for d in original] == [d["chunk_id"] for d in nuevo]
print("Equivalencia con el baseline comprobada en dos preguntas.")

paso0 = evaluacion.medir_retrieval(golden, "0_denso", DIRECTORIO,
                                  reutilizar=REUTILIZAR_RANKINGS)
display(pd.DataFrame([evaluacion.resumir_retrieval(paso0, golden)]))
detalle0 = pd.DataFrame(evaluacion.detalle_retrieval(paso0, golden)).merge(
    pd.DataFrame(con_ancla)[["id", "pregunta"]], on="id", validate="one_to_one")
with pd.option_context("display.max_colwidth", None):
    display(detalle0[["id", "pregunta", "item", "rango", "n_relevantes", "presente"]])
print("Consulta original, sin filtros. Recall@5:", evaluacion.recall_at_k(paso0, golden))


c:\Users\Fernando Dapena T\Desktop\Agentes\env_agentes\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3608.34it/s]


Equivalencia con el baseline comprobada en dos preguntas.


,n,no_indexables,mrr@10,ms_busqueda,ms_reescritura,fallos_reescritura,reintentos_reescritura,usd_reescritura,aciertos@1,recall@1,aciertos@3,recall@3,aciertos@5,recall@5,aciertos@10,recall@10
0,13,0,0.038462,16.171,0.0,0,0,0.0,0/13,0.0,0/13,0.0,2/13,0.153846,2/13,0.153846


,id,pregunta,item,rango,n_relevantes,presente
0,g3-008,"Según su 10-K de FY2025, ¿qué dice Microsoft sobre el uso indebido de sus sistemas de IA por parte de terceros?",1A,NaN,1,True
1,g3-009,¿Qué riesgo declara NVIDIA en FY2025 por depender de un número reducido de proveedores de fabricación?,1A,NaN,1,True
2,g3-010,¿Cómo describe Apple su exposición al riesgo de tipo de cambio en FY2024?,7A,NaN,1,True
3,g3-011,¿Qué dice la dirección de Amazon en FY2025 sobre la evolución de AWS?,7,4.0,1,True
4,g3-012,¿Qué riesgo regulatorio en materia de competencia declara Alphabet en FY2025?,1A,NaN,1,True
5,g3-013,¿Qué indica Meta en sus estados financieros de FY2024 sobre sus obligaciones contractuales o arrendamientos?,8,NaN,1,True
6,g3-014,"¿Cómo evolucionó los ingresos totales de MSFT entre los ejercicios fiscales 2024 y 2025, y qué explica la dirección al respecto?",7,NaN,1,True
7,g3-015,"¿Cómo evolucionó los ingresos totales de NVDA entre los ejercicios fiscales 2024 y 2025, y qué explica la dirección al respecto?",7,NaN,1,True
8,g3-016,"¿Cómo evolucionó los ingresos totales de GOOGL entre los ejercicios fiscales 2024 y 2025, y qué explica la dirección al respecto?",7,NaN,1,True
9,g3-017,"¿Cómo evolucionó el gasto en I+D de META entre los ejercicios fiscales 2024 y 2025, y qué explica la dirección al respecto?",7,NaN,1,True


Consulta original, sin filtros. Recall@5: 0.15384615384615385


## 2. Paso 1: filtro por metadatos

Guía: [11 §5](../docs/11_skill_mejora_retrieval.md)

In [4]:
# Los filtros oráculo vienen del golden: miden el potencial del filtro,
# no la capacidad del agente de inferir empresa, ejercicio y sección.
paso1 = evaluacion.medir_retrieval(golden, "1_filtro", DIRECTORIO,
                                  reutilizar=REUTILIZAR_RANKINGS)
display(pd.DataFrame([
    {"paso": "0_denso", **evaluacion.resumir_retrieval(paso0, golden)},
    {"paso": "1_filtro", **evaluacion.resumir_retrieval(paso1, golden)},
]))
meta = retrieval._leer_metadatos_cache()
candidatos = pd.DataFrame([
    {"id": p["id"], **evaluacion.filtros_oraculo(p),
     "n_candidatos": len(retrieval.candidatos(**evaluacion.filtros_oraculo(p)))}
    for p in con_ancla
])
candidatos["aleatorio@5"] = candidatos["n_candidatos"].map(
    lambda n: min(1, 5 / n) if n else 0)
display(candidatos)
print("Con búsqueda exacta, filtrar antes o después de ordenar TODO da el mismo top-k.")
print("El filtro sí debe preceder al recorte top-20 del híbrido.")


,paso,n,no_indexables,mrr@10,ms_busqueda,ms_reescritura,fallos_reescritura,reintentos_reescritura,usd_reescritura,aciertos@1,recall@1,aciertos@3,recall@3,aciertos@5,recall@5,aciertos@10,recall@10
0,0_denso,13,0,0.038462,16.171000,0.0,0,0,0.0,0/13,0.000000,0/13,0.000000,2/13,0.153846,2/13,0.153846
1,1_filtro,13,0,0.296154,18.126769,0.0,0,0,0.0,3/13,0.230769,4/13,0.307692,4/13,0.307692,7/13,0.538462


,id,ticker,fiscal_year,item,n_candidatos,aleatorio@5
0,g3-008,MSFT,2025,1A,32,0.156250
1,g3-009,NVDA,2025,1A,49,0.102041
2,g3-010,AAPL,2024,7A,2,1.000000
3,g3-011,AMZN,2025,7,25,0.200000
4,g3-012,GOOGL,2025,1A,39,0.128205
5,g3-013,META,2024,8,75,0.066667
6,g3-014,MSFT,2025,7,26,0.192308
7,g3-015,NVDA,2025,7,23,0.217391
8,g3-016,GOOGL,2025,7,29,0.172414
9,g3-017,META,2025,7,36,0.138889


Con búsqueda exacta, filtrar antes o después de ordenar TODO da el mismo top-k.
El filtro sí debe preceder al recorte top-20 del híbrido.


## 3. Paso 2: BM25 + denso con RRF

Guía: [11 §6](../docs/11_skill_mejora_retrieval.md)

In [5]:
# BM25 usa IDF global, la misma tokenización en corpus y consultas,
# y descarta puntuaciones <= 0. RRF combina posiciones con pesos iguales.
paso2 = evaluacion.medir_retrieval(golden, "2_bm25", DIRECTORIO,
                                  reutilizar=REUTILIZAR_RANKINGS)
solo_bm25 = evaluacion.medir_retrieval(golden, "d_solo_bm25", DIRECTORIO,
                                       reutilizar=REUTILIZAR_RANKINGS)
display(pd.DataFrame([
    {"paso": "1_filtro", **evaluacion.resumir_retrieval(paso1, golden)},
    {"paso": "2_bm25", **evaluacion.resumir_retrieval(paso2, golden)},
    {"paso": "d_solo_bm25", **evaluacion.resumir_retrieval(solo_bm25, golden)},
]))
ruta_techos = DIRECTORIO / "techos.jsonl"
if REUTILIZAR_RANKINGS and ruta_techos.is_file():
    techos = evaluacion.cargar_golden(ruta_techos)
else:
    techos = evaluacion.techos_retrieval(golden)
    evaluacion.guardar_golden(techos, ruta_techos)
display(pd.DataFrame(techos))
print("Las preguntas siguen en español: BM25 puede aportar poco hasta la reescritura.")
print("Si el ancla no está en ninguno de los dos top-20, RRF no puede recuperarla.")


,paso,n,no_indexables,mrr@10,ms_busqueda,ms_reescritura,fallos_reescritura,reintentos_reescritura,usd_reescritura,aciertos@1,recall@1,aciertos@3,recall@3,aciertos@5,recall@5,aciertos@10,recall@10
0,1_filtro,13,0,0.296154,18.126769,0.0,0,0,0.0,3/13,0.230769,4/13,0.307692,4/13,0.307692,7/13,0.538462
1,2_bm25,13,0,0.241758,18.951769,0.0,0,0,0.0,2/13,0.153846,3/13,0.230769,5/13,0.384615,6/13,0.461538
2,d_solo_bm25,13,0,0.135989,0.396615,0.0,0,0,0.0,1/13,0.076923,1/13,0.076923,2/13,0.153846,7/13,0.538462


,id,item,n_candidatos,aleatorio@5,techo_filtro,hit_denso@20,hit_bm25@20,techo_hibrido
0,g3-008,1A,32,0.156250,True,True,False,True
1,g3-009,1A,49,0.102041,True,True,False,True
2,g3-010,7A,2,1.000000,True,True,True,True
3,g3-011,7,25,0.200000,True,True,True,True
4,g3-012,1A,39,0.128205,True,True,True,True
5,g3-013,8,75,0.066667,True,True,False,True
6,g3-014,7,26,0.192308,True,False,False,False
7,g3-015,7,23,0.217391,True,True,True,True
8,g3-016,7,29,0.172414,True,True,True,True
9,g3-017,7,36,0.138889,True,True,True,True


Las preguntas siguen en español: BM25 puede aportar poco hasta la reescritura.
Si el ancla no está en ninguno de los dos top-20, RRF no puede recuperarla.


## 4. Paso 3: reescritura de la consulta con el LLM

Guía: [11 §7](../docs/11_skill_mejora_retrieval.md)

In [6]:
# Única celda que puede llamar a OpenRouter: solo con EJECUTAR=True
# y para preguntas sin caché válida; los fallos se reintentan. Nunca se imprime la clave.
# Usa 1-3 consultas en inglés y filtros del modelo corregidos con entidades explícitas.
# Las comparativas buscan ambos ejercicios y fusionan sus rankings.
paso3 = None
try:
    paso3 = evaluacion.medir_retrieval(
        golden, "3_reescritura", DIRECTORIO,
        permitir_api=EJECUTAR, reutilizar=REUTILIZAR_RANKINGS,
    )
    rw_oraculo = evaluacion.medir_retrieval(
        golden, "d_rw_oraculo", DIRECTORIO,
        permitir_api=EJECUTAR, reutilizar=REUTILIZAR_RANKINGS,
    )
except retrieval.ReescrituraPendiente as exc:
    print(str(exc))
    print("Puedes continuar: la tabla final indicará que falta el paso 3.")
else:
    display(pd.DataFrame([{
        "id": f["id"], **f["reescritura"]["busqueda"],
        "fallo": f["reescritura"]["fallo"], "error": f["reescritura"]["error"],
        "ms": f["reescritura"]["ms"], "usd": f["reescritura"]["usd"],
        "intentos": f["reescritura"].get("intentos"),
        "ajustes_explicitos": f["reescritura"].get("ajustes_filtros"),
        "filtros_llm": f.get("filtros_llm"),
        "filtros": f["filtros"],
    } for f in paso3]))
    display(pd.DataFrame([
        {"paso": "3_reescritura", **evaluacion.resumir_retrieval(paso3, golden)},
        {"paso": "d_rw_oraculo", **evaluacion.resumir_retrieval(rw_oraculo, golden)},
    ]))
    print("USD vacío significa coste no comunicado; no se sustituye por cero.")
    print("El tiempo de reescritura se guarda aparte del tiempo de búsqueda.")
    print("En comparativas, oráculo usa el FY reciente; el paso 3 busca ambos FY.")


,id,consultas,ticker,fiscal_years,item,fallo,error,ms,usd,intentos,ajustes_explicitos,filtros_llm,filtros
0,g3-008,"[Según su 10-K de FY2025, ¿qué dice Microsoft ...",MSFT,[2025],None,True,BadRequestResponseError,2068.78,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
1,g3-009,[¿Qué riesgo declara NVIDIA en FY2025 por depe...,NVDA,[2025],None,True,BadRequestResponseError,1579.43,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
2,g3-010,[¿Cómo describe Apple su exposición al riesgo ...,AAPL,[2024],None,True,BadRequestResponseError,1198.18,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
3,g3-011,[¿Qué dice la dirección de Amazon en FY2025 so...,AMZN,[2025],None,True,BadRequestResponseError,1387.88,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
4,g3-012,[¿Qué riesgo regulatorio en materia de compete...,GOOGL,[2025],None,True,BadRequestResponseError,1050.25,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
5,g3-013,[¿Qué indica Meta en sus estados financieros d...,META,[2024],None,True,BadRequestResponseError,1086.81,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
6,g3-014,[¿Cómo evolucionó los ingresos totales de MSFT...,MSFT,"[2024, 2025]",None,True,BadRequestResponseError,1597.50,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
7,g3-015,[¿Cómo evolucionó los ingresos totales de NVDA...,NVDA,"[2024, 2025]",None,True,BadRequestResponseError,1192.78,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
8,g3-016,[¿Cómo evolucionó los ingresos totales de GOOG...,GOOGL,"[2024, 2025]",None,True,BadRequestResponseError,1043.41,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."
9,g3-017,[¿Cómo evolucionó el gasto en I+D de META entr...,META,"[2024, 2025]",None,True,BadRequestResponseError,6648.92,None,2,{},None,"{'ticker': 'ok', 'fiscal_years': 'ok', 'item':..."


,paso,n,no_indexables,mrr@10,ms_busqueda,ms_reescritura,fallos_reescritura,reintentos_reescritura,usd_reescritura,aciertos@1,recall@1,aciertos@3,recall@3,aciertos@5,recall@5,aciertos@10,recall@10
0,3_reescritura,13,0,0.036630,19.027462,1759.623077,13,13,None,0/13,0.000000,1/13,0.076923,1/13,0.076923,2/13,0.153846
1,d_rw_oraculo,13,0,0.241758,17.391923,2608.963846,13,13,None,2/13,0.153846,3/13,0.230769,5/13,0.384615,6/13,0.461538


USD vacío significa coste no comunicado; no se sustituye por cero.
El tiempo de reescritura se guarda aparte del tiempo de búsqueda.
En comparativas, oráculo usa el FY reciente; el paso 3 busca ambos FY.


## 5. Tabla de la escalera y configuración final

Guía: [11 §8](../docs/11_skill_mejora_retrieval.md) · [11 §10](../docs/11_skill_mejora_retrieval.md)

In [7]:
# Regenera CSV y JSON desde los rankings guardados, sin nuevas llamadas.
# No se generan ficheros ni celdas Markdown.
tablas = evaluacion.exportar_retrieval(golden, DIRECTORIO)
for nombre in ("escalera", "por_item", "filtros", "cambios"):
    print(nombre)
    display(tablas[nombre])

resumen = json.loads((DIRECTORIO / "resumen.json").read_text(encoding="utf-8"))
print("Escalera completa sin fallos:", resumen["completo"], "| Pendientes:", resumen["pendientes"],
      "| Reescrituras fallidas:", resumen.get("fallos_reescritura", 0))
print("Configuración y resultados:", DIRECTORIO)

# Herramienta final lista para conectarla al sistema final del notebook 07.
# La instancia original search_filings y TOOLS conservan el baseline del 04.
search_filings_final = herramientas.crear_search_filings("final")
tools_retrieval_final = [
    search_filings_final if t.name == "search_filings" else t for t in herramientas.TOOLS
]
assert search_filings_final.name == herramientas.search_filings.name
assert list(search_filings_final.args) == list(herramientas.search_filings.args)
print("Herramienta híbrida preparada:", search_filings_final.name,
      "| argumentos:", list(search_filings_final.args))

# Diagnóstico dentro del agente solo si ya existen sus predicciones.
# Este notebook no ejecuta el agente ni necesita resultados del 04.
if RUTA_PREDICCIONES_AGENTE.is_file():
    predicciones = evaluacion.cargar_golden(RUTA_PREDICCIONES_AGENTE)
    diagnostico = evaluacion.diagnosticar_retrieval_agente(predicciones, golden)
    diagnostico.to_csv(DIRECTORIO / "diagnostico_agente.csv", index=False)
    display(diagnostico)
else:
    print("Diagnóstico del agente pendiente: todavía no hay predicciones en",
          RUTA_PREDICCIONES_AGENTE)


escalera


,paso,n,no_indexables,mrr@10,ms_busqueda,ms_reescritura,fallos_reescritura,reintentos_reescritura,usd_reescritura,aciertos@1,recall@1,aciertos@3,recall@3,aciertos@5,recall@5,aciertos@10,recall@10
0,0_denso,13,0,0.038462,16.171000,0.000000,0,0,0.0,0/13,0.000000,0/13,0.000000,2/13,0.153846,2/13,0.153846
1,1_filtro,13,0,0.296154,18.126769,0.000000,0,0,0.0,3/13,0.230769,4/13,0.307692,4/13,0.307692,7/13,0.538462
2,2_bm25,13,0,0.241758,18.951769,0.000000,0,0,0.0,2/13,0.153846,3/13,0.230769,5/13,0.384615,6/13,0.461538
3,3_reescritura,13,0,0.036630,19.027462,1759.623077,13,13,NaN,0/13,0.000000,1/13,0.076923,1/13,0.076923,2/13,0.153846
4,d_solo_bm25,13,0,0.135989,0.396615,0.000000,0,0,0.0,1/13,0.076923,1/13,0.076923,2/13,0.153846,7/13,0.538462
5,d_rw_oraculo,13,0,0.241758,17.391923,2608.963846,13,13,NaN,2/13,0.153846,3/13,0.230769,5/13,0.384615,6/13,0.461538


por_item


,paso,item,aciertos@5,recall@5
0,0_denso,1A,0/3,0.000000
1,0_denso,7,1/6,0.166667
2,0_denso,7A,0/1,0.000000
3,0_denso,8,1/3,0.333333
4,1_filtro,1A,0/3,0.000000
5,1_filtro,7,2/6,0.333333
6,1_filtro,7A,1/1,1.000000
7,1_filtro,8,1/3,0.333333
8,2_bm25,1A,1/3,0.333333
9,2_bm25,7,2/6,0.333333


filtros


,campo,estado,n,porcentaje
0,ticker,ok,13,100.0
1,ticker,ausente,0,0.0
2,ticker,erroneo,0,0.0
3,fiscal_years,ok,13,100.0
4,fiscal_years,ausente,0,0.0
5,fiscal_years,erroneo,0,0.0
6,item,ok,0,0.0
7,item,ausente,13,100.0
8,item,erroneo,0,0.0


cambios


,antes,despues,id,rango_antes,rango_despues,cambio
0,0_denso,1_filtro,g3-008,NaN,11.0,igual
1,0_denso,1_filtro,g3-009,NaN,10.0,igual
2,0_denso,1_filtro,g3-010,NaN,1.0,ganada
3,0_denso,1_filtro,g3-011,4.0,1.0,igual
4,0_denso,1_filtro,g3-012,NaN,8.0,igual
5,0_denso,1_filtro,g3-013,NaN,16.0,igual
6,0_denso,1_filtro,g3-014,NaN,NaN,igual
7,0_denso,1_filtro,g3-015,NaN,20.0,igual
8,0_denso,1_filtro,g3-016,NaN,16.0,igual
9,0_denso,1_filtro,g3-017,NaN,8.0,igual


Escalera completa sin fallos: False | Pendientes: [] | Reescrituras fallidas: 13
Configuración y resultados: C:\Users\Fernando Dapena T\Desktop\Agentes\10k-financial-agent\resultados\retrieval\7d0a40294471
Herramienta híbrida preparada: search_filings | argumentos: ['query', 'ticker', 'fiscal_year', 'item', 'k']


,id,llamada,recall_agente,ticker,fiscal_years,item
0,g3-008,1,True,ok,ok,ok
1,g3-009,1,True,ok,ok,ok
2,g3-010,1,True,ok,ok,ok
3,g3-011,1,True,ok,ausente,ausente
4,g3-011,2,True,ok,ausente,ausente
5,g3-011,3,True,ok,ausente,ausente
6,g3-011,4,True,ok,ausente,ausente
7,g3-011,5,True,ok,ausente,ausente
8,g3-012,1,True,ok,ok,ok
9,g3-013,1,True,erroneo,ausente,ausente


## 6. Pruebas adicionales (locales) en busca de una mejor solución · Consultas en inglés, sin item y con item

Se comparan tres variantes sobre las mismas 13 preguntas con ancla: limpieza general con denso, consulta breve con BM25 y consulta reformulada con denso. **Cada variante se ejecuta en dos escenarios:**

- **Sin item:** empresa y ejercicio extraídos de la pregunta original.
- **Con item del golden:** los mismos filtros anteriores, añadiendo únicamente la sección correcta del golden (oráculo).

Así se puede comparar cada método en igualdad de condiciones y medir qué aporta conocer la sección. En ambos escenarios se conserva la consulta literal como control con el mismo buscador y filtros, además de la referencia densa sin filtros.

**Cómo ejecutarlo.** Empieza por la celda de «Las 13 preguntas para la prueba» y continúa por 6.1, 6.2 y 6.3. La primera celda calcula las dos pruebas de limpieza; 6.1 solo muestra sus resultados. No necesitas ejecutar los pasos 0–3. Estas pruebas son locales y no llaman a OpenRouter ni dependen de `EJECUTAR`. BGE debe estar descargado. Las consultas manuales están en [consultas.json](../resultados/experimentos/ingles_20260917/consultas.json).

**Cómo decidir.** Compara primero las filas **sin item** usando recall@5; MRR@10 informa de la posición de la evidencia. Las filas **con item** miden la ayuda de conocer la sección, no la capacidad de inferirla. Las tablas se recalculan, sin porcentajes prefijados.

**Límites.** Traducción y reformulación son manuales. En comparativas se conserva el ejercicio más reciente: esto no comprueba recuperar evidencia de ambos años. Son resultados exploratorios sobre el mismo golden usado para preparar las consultas. Encontrar el ancla no garantiza responder correctamente. Un rango vacío significa que el ancla no apareció en el top-20.

### Las 13 preguntas para la prueba 

In [8]:
import sys
import pathlib
import pandas as pd
from IPython.display import display

RAIZ_PRUEBAS = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
if str(RAIZ_PRUEBAS / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ_PRUEBAS / "src"))

from agente10k.pruebas_retrieval import probar_variante

# Calcula ambos escenarios una sola vez; 6.1 solo muestra los resultados.
prueba_limpieza = probar_variante("limpieza_densa")
prueba_limpieza_con_item = probar_variante("limpieza_densa", usar_item=True)

with pd.option_context("display.max_colwidth", None):
    display(prueba_limpieza["detalle"][
        ["id", "consulta_literal", "consulta_prueba"]
    ])

,id,consulta_literal,consulta_prueba
0,g3-008,"According to its FY2025 10-K, what does Microsoft say about misuse of its AI systems by third parties?",k misuse ai systems third parties
1,g3-009,What risk does NVIDIA report in FY2025 from depending on a small number of manufacturing suppliers?,risk depending small number manufacturing suppliers
2,g3-010,How does Apple describe its exposure to foreign exchange risk in FY2024?,exposure foreign exchange risk
3,g3-011,What does Amazon management say in FY2025 about the evolution of AWS?,evolution aws
4,g3-012,What competition-related regulatory risk does Alphabet report in FY2025?,competition related regulatory risk
5,g3-013,What does Meta indicate in its FY2024 financial statements about its contractual obligations or leases?,contractual obligations leases
6,g3-014,"How did MSFT total revenue evolve between fiscal years 2024 and 2025, and what does management explain about it?",revenue evolve
7,g3-015,"How did NVDA total revenue evolve between fiscal years 2024 and 2025, and what does management explain about it?",revenue evolve
8,g3-016,"How did GOOGL total revenue evolve between fiscal years 2024 and 2025, and what does management explain about it?",revenue evolve
9,g3-017,"How did META research and development expense evolve between fiscal years 2024 and 2025, and what does management explain about it?",research development expense evolve


### 6.1. Denso con limpieza general

**Qué se hace.** Se parte de la traducción manual al inglés y se eliminan palabras poco útiles, nombres de empresa y años con la misma regla para todas las preguntas. Empresa y ejercicio se extraen de la pregunta original.

**Dos pruebas.** Se busca primero sin sección y después con el item del golden. La consulta, el buscador, la empresa y el ejercicio permanecen iguales: solo cambia la sección conocida.

**Controles.** Denso literal sin filtros, denso literal con los filtros del escenario y denso con consulta limpiada. Permiten distinguir el efecto de limpiar la consulta del de limitar la búsqueda.

### Ejemplo

1. Partir de la pregunta original en español y extraer la empresa y el ejercicio fiscal para usarlos como filtros.
2. Traducir manualmente la pregunta del español al inglés, el idioma de los documentos.
3. Limpiar esa traducción eliminando empresa, años y palabras poco útiles, con las mismas reglas para todas las preguntas.
4. Convertir la consulta limpia en un embedding y recuperar los fragmentos más similares semánticamente, filtrando por empresa y ejercicio, sin item.
5. Repetir la búsqueda añadiendo únicamente el item correcto del golden. Así se mide cuánto ayuda conocer previamente la sección.

In [9]:
query = "revenue evolve"
ticker = "MSFT"
fiscal_year = 2025

# Solo en la segunda prueba se añade la sección conocida.
item_oraculo = "7"

In [10]:
# Muestra los dos escenarios calculados arriba; no repite búsquedas.
for resultado in (prueba_limpieza, prueba_limpieza_con_item):
    print(resultado["resumen"]["escenario"].iloc[0])
    display(resultado["resumen"].style.format({"recall@5": "{:.1%}", "mrr@10": "{:.3f}"}))
    with pd.option_context("display.max_colwidth", 100):
        display(resultado["detalle"][
            ["id", "consulta_prueba", "ticker", "fiscal_year", "item",
             "rango_literal", "rango_prueba", "acierto@5"]
        ])

Sin item


,escenario,prueba,backend,filtros,aciertos@5,recall@5,aciertos@10,mrr@10
0,Sin item,Denso literal sin filtros,denso,Ninguno,5/13,38.5%,6/13,0.198
1,Sin item,Consulta literal: mismo backend y filtros,denso,Empresa y FY extraídos de la pregunta,6/13,46.2%,7/13,0.381
2,Sin item,Denso + limpieza general,denso,Empresa y FY extraídos de la pregunta,10/13,76.9%,10/13,0.515


,id,consulta_prueba,ticker,fiscal_year,item,rango_literal,rango_prueba,acierto@5
0,g3-008,k misuse ai systems third parties,MSFT,2025,None,8.0,4.0,True
1,g3-009,risk depending small number manufacturing suppliers,NVDA,2025,None,NaN,13.0,False
2,g3-010,exposure foreign exchange risk,AAPL,2024,None,2.0,1.0,True
3,g3-011,evolution aws,AMZN,2025,None,1.0,2.0,True
4,g3-012,competition related regulatory risk,GOOGL,2025,None,NaN,NaN,False
5,g3-013,contractual obligations leases,META,2024,None,3.0,1.0,True
6,g3-014,revenue evolve,MSFT,2025,None,NaN,1.0,True
7,g3-015,revenue evolve,NVDA,2025,None,14.0,2.0,True
8,g3-016,revenue evolve,GOOGL,2025,None,12.0,5.0,True
9,g3-017,research development expense evolve,META,2025,None,1.0,1.0,True


Con item del golden


,escenario,prueba,backend,filtros,aciertos@5,recall@5,aciertos@10,mrr@10
0,Con item del golden,Denso literal sin filtros,denso,Ninguno,5/13,38.5%,6/13,0.198
1,Con item del golden,Consulta literal: mismo backend y filtros,denso,Empresa y FY extraídos de la pregunta + item del golden (oráculo),7/13,53.8%,10/13,0.456
2,Con item del golden,Denso + limpieza general,denso,Empresa y FY extraídos de la pregunta + item del golden (oráculo),10/13,76.9%,10/13,0.603


,id,consulta_prueba,ticker,fiscal_year,item,rango_literal,rango_prueba,acierto@5
0,g3-008,k misuse ai systems third parties,MSFT,2025,1A,6,4.0,True
1,g3-009,risk depending small number manufacturing suppliers,NVDA,2025,1A,11,12.0,False
2,g3-010,exposure foreign exchange risk,AAPL,2024,7A,1,1.0,True
3,g3-011,evolution aws,AMZN,2025,7,1,1.0,True
4,g3-012,competition related regulatory risk,GOOGL,2025,1A,11,NaN,False
5,g3-013,contractual obligations leases,META,2024,8,3,1.0,True
6,g3-014,revenue evolve,MSFT,2025,7,10,1.0,True
7,g3-015,revenue evolve,NVDA,2025,7,5,1.0,True
8,g3-016,revenue evolve,GOOGL,2025,7,8,3.0,True
9,g3-017,research development expense evolve,META,2025,7,1,1.0,True


### 6.2. BM25 con consultas breves de palabras clave

**Qué se hace.** Se usa una consulta manual breve en inglés, por ejemplo `total revenue growth drivers`. BM25 busca coincidencias léxicas con vocabulario financiero. Mantiene el IDF global, la misma tokenización de consulta y corpus y el descarte de puntuaciones no positivas.

**Dos pruebas.** Empresa y ejercicio se extraen de la pregunta, igual que en 6.1. Se ejecuta sin item y se repite añadiendo el item del golden. No se toman empresa ni año del golden para buscar.

**Controles.** Además del denso literal sin filtros, se muestran denso literal y BM25 literal con los filtros de cada escenario. La comparación con BM25 breve aísla el efecto de las palabras elegidas.

### Ejemplo

1. Partir de la pregunta original en español y extraer la empresa y el ejercicio fiscal como filtros.
2. Preparar manualmente una consulta breve en inglés que recoja los conceptos principales. Se traduce y se resume la intención, sin conservar necesariamente la pregunta completa.
3. Por ejemplo, una pregunta sobre la evolución de ingresos se convierte en `total revenue growth drivers`.
4. Buscar con BM25 los fragmentos con mejores coincidencias de palabras, filtrando por empresa y ejercicio, sin item. Este buscador no utiliza embeddings.
5. Repetir con la misma consulta y los mismos filtros, añadiendo únicamente el item correcto del golden.

In [11]:
query = "total revenue growth drivers"
ticker = "MSFT"
fiscal_year = 2025
# Se añade únicamente en la segunda prueba.
item_oraculo = "7"

In [12]:
prueba_bm25 = probar_variante("bm25_breve")
prueba_bm25_con_item = probar_variante("bm25_breve", usar_item=True)

for resultado in (prueba_bm25, prueba_bm25_con_item):
    print(resultado["resumen"]["escenario"].iloc[0])
    display(resultado["resumen"].style.format({"recall@5": "{:.1%}", "mrr@10": "{:.3f}"}))
    with pd.option_context("display.max_colwidth", 100):
        display(resultado["detalle"][
            ["id", "consulta_prueba", "ticker", "fiscal_year", "item",
             "rango_literal", "rango_prueba", "acierto@5"]
        ])

Sin item


,escenario,prueba,backend,filtros,aciertos@5,recall@5,aciertos@10,mrr@10
0,Sin item,Denso literal sin filtros,denso,Ninguno,5/13,38.5%,6/13,0.198
1,Sin item,Denso literal con los mismos filtros,denso,Empresa y FY extraídos de la pregunta,6/13,46.2%,7/13,0.381
2,Sin item,Consulta literal: mismo backend y filtros,bm25,Empresa y FY extraídos de la pregunta,4/13,30.8%,5/13,0.152
3,Sin item,BM25 + consulta breve,bm25,Empresa y FY extraídos de la pregunta,11/13,84.6%,13/13,0.330


,id,consulta_prueba,ticker,fiscal_year,item,rango_literal,rango_prueba,acierto@5
0,g3-008,third party misuse of artificial intelligence systems,MSFT,2025,None,12.0,2,True
1,g3-009,dependence on a limited number of manufacturing suppliers,NVDA,2025,None,15.0,5,True
2,g3-010,foreign exchange risk exposure,AAPL,2024,None,2.0,2,True
3,g3-011,AWS business performance revenue growth,AMZN,2025,None,NaN,5,True
4,g3-012,competition regulation antitrust risks,GOOGL,2025,None,12.0,4,True
5,g3-013,contractual obligations and lease commitments,META,2024,None,2.0,3,True
6,g3-014,total revenue growth drivers,MSFT,2025,None,NaN,3,True
7,g3-015,total revenue growth drivers,NVDA,2025,None,NaN,4,True
8,g3-016,total revenue growth drivers,GOOGL,2025,None,NaN,7,False
9,g3-017,research and development expense change drivers,META,2025,None,7.0,1,True


Con item del golden


,escenario,prueba,backend,filtros,aciertos@5,recall@5,aciertos@10,mrr@10
0,Con item del golden,Denso literal sin filtros,denso,Ninguno,5/13,38.5%,6/13,0.198
1,Con item del golden,Denso literal con los mismos filtros,denso,Empresa y FY extraídos de la pregunta + item del golden (oráculo),7/13,53.8%,10/13,0.456
2,Con item del golden,Consulta literal: mismo backend y filtros,bm25,Empresa y FY extraídos de la pregunta + item del golden (oráculo),6/13,46.2%,9/13,0.263
3,Con item del golden,BM25 + consulta breve,bm25,Empresa y FY extraídos de la pregunta + item del golden (oráculo),12/13,92.3%,13/13,0.471


,id,consulta_prueba,ticker,fiscal_year,item,rango_literal,rango_prueba,acierto@5
0,g3-008,third party misuse of artificial intelligence systems,MSFT,2025,1A,8.0,2,True
1,g3-009,dependence on a limited number of manufacturing suppliers,NVDA,2025,1A,9.0,5,True
2,g3-010,foreign exchange risk exposure,AAPL,2024,7A,1.0,1,True
3,g3-011,AWS business performance revenue growth,AMZN,2025,7,17.0,3,True
4,g3-012,competition regulation antitrust risks,GOOGL,2025,1A,4.0,1,True
5,g3-013,contractual obligations and lease commitments,META,2024,8,2.0,2,True
6,g3-014,total revenue growth drivers,MSFT,2025,7,NaN,3,True
7,g3-015,total revenue growth drivers,NVDA,2025,7,14.0,3,True
8,g3-016,total revenue growth drivers,GOOGL,2025,7,11.0,5,True
9,g3-017,research and development expense change drivers,META,2025,7,2.0,1,True


### 6.3. Denso con consultas reformuladas hacia la explicación

**Qué se hace.** Se sustituye la traducción literal por una consulta manual centrada en la explicación buscada. Por ejemplo, para ingresos: `revenue increased decreased reasons management discussion`. Se conserva el mismo modelo de embeddings.

**Dos pruebas.** Empresa y ejercicio se extraen de la pregunta original, igual que en 6.1 y 6.2. Se busca sin item y después añadiendo el item del golden, con la misma consulta.

**Controles.** Se compara con el denso literal sin filtros y con el denso literal que usa los filtros de cada escenario. El detalle muestra tanto mejoras como regresiones por pregunta. La tabla final reúne las seis combinaciones y no selecciona automáticamente una configuración para el agente.

### Ejemplo

1. Partir de la pregunta original en español y extraer la empresa y el ejercicio fiscal como filtros.
2. Reformular manualmente su intención en inglés, incluyendo el tema y el tipo de explicación buscada. No se trata solo de traducir literalmente.
3. Por ejemplo, para buscar las causas de una variación de ingresos: `revenue increased decreased reasons management discussion`.
4. Convertir la reformulación en un embedding y buscar fragmentos similares semánticamente, filtrando por empresa y ejercicio, sin item.
5. Repetir añadiendo únicamente el item correcto del golden, manteniendo la consulta y el modelo de embeddings.

In [13]:
query_original = "How did MSFT total revenue evolve between fiscal years 2024 and 2025, and what does management explain about it?"
query_reformulada = "revenue increased decreased reasons management discussion"

# Mismos filtros para ambas consultas
ticker = "MSFT"
fiscal_year = 2025
# Se añade únicamente en la segunda prueba.
item_oraculo = "7"

In [14]:
prueba_denso = probar_variante("denso_reformulado")
prueba_denso_con_item = probar_variante("denso_reformulado", usar_item=True)

for resultado in (prueba_denso, prueba_denso_con_item):
    print(resultado["resumen"]["escenario"].iloc[0])
    display(resultado["resumen"].style.format({"recall@5": "{:.1%}", "mrr@10": "{:.3f}"}))
    with pd.option_context("display.max_colwidth", 100):
        display(resultado["detalle"][
            ["id", "consulta_prueba", "ticker", "fiscal_year", "item",
             "rango_literal", "rango_prueba", "acierto@5"]
        ])

comparacion_pruebas = pd.concat(
    [p["resumen"].tail(1) for p in (
        prueba_limpieza, prueba_limpieza_con_item,
        prueba_bm25, prueba_bm25_con_item,
        prueba_denso, prueba_denso_con_item,
    )],
    ignore_index=True,
)
print("Seis combinaciones: mismos filtros de empresa y FY; solo cambia conocer el item.")
display(comparacion_pruebas.style.format({"recall@5": "{:.1%}", "mrr@10": "{:.3f}"}))

Sin item


,escenario,prueba,backend,filtros,aciertos@5,recall@5,aciertos@10,mrr@10
0,Sin item,Denso literal sin filtros,denso,Ninguno,5/13,38.5%,6/13,0.198
1,Sin item,Consulta literal: mismo backend y filtros,denso,Empresa y FY extraídos de la pregunta,6/13,46.2%,7/13,0.381
2,Sin item,Denso + consulta reformulada,denso,Empresa y FY extraídos de la pregunta,8/13,61.5%,11/13,0.506


,id,consulta_prueba,ticker,fiscal_year,item,rango_literal,rango_prueba,acierto@5
0,g3-008,artificial intelligence services abuse risks and safeguards,MSFT,2025,None,8.0,3.0,True
1,g3-009,supplier concentration manufacturing supply chain risks,NVDA,2025,None,NaN,6.0,False
2,g3-010,foreign currency exchange rate risk management,AAPL,2024,None,2.0,1.0,True
3,g3-011,Amazon Web Services sales growth drivers management discussion,AMZN,2025,None,1.0,2.0,True
4,g3-012,antitrust law competition regulatory proceedings,GOOGL,2025,None,NaN,6.0,False
5,g3-013,operating leases obligations financial statement notes,META,2024,None,3.0,1.0,True
6,g3-014,revenue increased decreased reasons management discussion,MSFT,2025,None,NaN,1.0,True
7,g3-015,revenue increased decreased reasons management discussion,NVDA,2025,None,14.0,6.0,False
8,g3-016,revenue increased decreased reasons management discussion,GOOGL,2025,None,12.0,12.0,False
9,g3-017,research and development costs increased decreased reasons,META,2025,None,1.0,1.0,True


Con item del golden


,escenario,prueba,backend,filtros,aciertos@5,recall@5,aciertos@10,mrr@10
0,Con item del golden,Denso literal sin filtros,denso,Ninguno,5/13,38.5%,6/13,0.198
1,Con item del golden,Consulta literal: mismo backend y filtros,denso,Empresa y FY extraídos de la pregunta + item del golden (oráculo),7/13,53.8%,10/13,0.456
2,Con item del golden,Denso + consulta reformulada,denso,Empresa y FY extraídos de la pregunta + item del golden (oráculo),12/13,92.3%,12/13,0.569


,id,consulta_prueba,ticker,fiscal_year,item,rango_literal,rango_prueba,acierto@5
0,g3-008,artificial intelligence services abuse risks and safeguards,MSFT,2025,1A,6,3.0,True
1,g3-009,supplier concentration manufacturing supply chain risks,NVDA,2025,1A,11,5.0,True
2,g3-010,foreign currency exchange rate risk management,AAPL,2024,7A,1,1.0,True
3,g3-011,Amazon Web Services sales growth drivers management discussion,AMZN,2025,7,1,2.0,True
4,g3-012,antitrust law competition regulatory proceedings,GOOGL,2025,1A,11,3.0,True
5,g3-013,operating leases obligations financial statement notes,META,2024,8,3,1.0,True
6,g3-014,revenue increased decreased reasons management discussion,MSFT,2025,7,10,1.0,True
7,g3-015,revenue increased decreased reasons management discussion,NVDA,2025,7,5,2.0,True
8,g3-016,revenue increased decreased reasons management discussion,GOOGL,2025,7,8,5.0,True
9,g3-017,research and development costs increased decreased reasons,META,2025,7,1,1.0,True


Seis combinaciones: mismos filtros de empresa y FY; solo cambia conocer el item.


,escenario,prueba,backend,filtros,aciertos@5,recall@5,aciertos@10,mrr@10
0,Sin item,Denso + limpieza general,denso,Empresa y FY extraídos de la pregunta,10/13,76.9%,10/13,0.515
1,Con item del golden,Denso + limpieza general,denso,Empresa y FY extraídos de la pregunta + item del golden (oráculo),10/13,76.9%,10/13,0.603
2,Sin item,BM25 + consulta breve,bm25,Empresa y FY extraídos de la pregunta,11/13,84.6%,13/13,0.330
3,Con item del golden,BM25 + consulta breve,bm25,Empresa y FY extraídos de la pregunta + item del golden (oráculo),12/13,92.3%,13/13,0.471
4,Sin item,Denso + consulta reformulada,denso,Empresa y FY extraídos de la pregunta,8/13,61.5%,11/13,0.506
5,Con item del golden,Denso + consulta reformulada,denso,Empresa y FY extraídos de la pregunta + item del golden (oráculo),12/13,92.3%,12/13,0.569


## 7. Pruebas de preguntas difíciles

Usamos la configuración elegida en cada apartado:

- **6.1 · Denso con limpieza:** «¿Qué riesgo declara NVIDIA en FY2025 por depender de pocos proveedores de fabricación?». Puede recuperar información general sobre proveedores y dejar fuera el fragmento esperado.
- **6.2 · BM25 con palabras clave:** «¿Cómo cambia el BPA de NVIDIA entre 2024 y 2025 y son comparables ambas cifras?». Las palabras clave pueden encontrar cifras de BPA sin recuperar la explicación sobre su comparabilidad.
- **6.3 · Denso con reformulación:** usamos la misma pregunta del BPA. La similitud de significado puede recuperar textos relacionados sin encontrar la explicación concreta que necesitamos.

Para cada método comparamos su consulta actual con una alternativa manual en ambos escenarios: sin item y con item del golden. Empresa y ejercicio se extraen siempre de la pregunta. Dentro de cada escenario se mantienen el mismo buscador y los mismos filtros.

**Cómo leer las tablas:** hay acierto si aparece el ancla esperada entre los primeros 5, 10 o 20 resultados. Fallar significa no recuperar esa evidencia concreta; no necesariamente que todos los fragmentos sean inútiles.

Estas pruebas son locales, sin agente ni API. Solo el item de la segunda prueba viene del golden. Las alternativas son exploratorias: no garantizan una mejora.

In [15]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Funciona desde la raíz del repositorio o desde notebooks/.
raiz = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(raiz / "src"))

from agente10k import config, evaluacion, retrieval
from agente10k.pruebas_retrieval import limpiar_consulta, extraer_filtros

golden = {
    p["id"]: p
    for p in evaluacion.cargar_golden(config.GOLDEN / "golden_propio.jsonl")
}
consultas = json.loads((
    config.RESULTADOS / "experimentos" / "ingles_20260917" / "consultas.json"
).read_text(encoding="utf-8"))

# Consulta actual: se carga la que ya utiliza cada apartado.
# Alternativa: otro intento manual, sin garantía de mejora.
casos = [
    ("6.1 · Denso + limpieza", "g3-009", 0,
     "dependence on limited third party manufacturers risks"),

    ("6.2 · BM25 + palabras clave", "g3-020", 1,
     "basic earnings per share prior year comparison adjusted shares"),

    ("6.3 · Denso + reformulación", "g3-020", 2,
     "comparability of basic earnings per share across years adjustments to number of shares"),
]

outcomes = {}

for metodo, pid, columna, alternativa in casos:
    pregunta = golden[pid]
    actual = consultas[pid][columna]

    if columna == 0:
        actual = limpiar_consulta(actual)
        alternativa = limpiar_consulta(alternativa)
    filtros_base = extraer_filtros(pregunta["pregunta"])

    buscar = retrieval.buscar_bm25 if columna == 1 else retrieval.buscar_denso
    filas = []

    for usar_item in (False, True):
        escenario = "Con item del golden" if usar_item else "Sin item"
        filtros = dict(filtros_base)
        if usar_item:
            filtros["item"] = pregunta["item_esperado"]
        for version, query in [
            ("Actual del apartado 6", actual),
            ("Alternativa manual", alternativa),
        ]:
            documentos = buscar(query, k=20, **filtros)
            ranking = [{
                "id": pid,
                "ranking": [d["chunk_id"] for d in documentos],
                "ms": 0,
            }]
            detalle = evaluacion.detalle_retrieval(ranking, [pregunta])[0]

            if not detalle["presente"]:
                raise ValueError(f"{pid}: el ancla no está indexada; revisa el caso.")

            rango = detalle["rango"]
            filas.append({
                "escenario": escenario,
                "item": filtros.get("item"),
                "versión": version,
                "query": query,
                "rango_ancla": rango if rango is not None else "Fuera del top-20",
                "acierto@5": rango is not None and rango <= 5,
                "acierto@10": rango is not None and rango <= 10,
                "acierto@20": rango is not None and rango <= 20,
                "outcome": (
                    "ACIERTO en top-5"
                    if rango is not None and rango <= 5
                    else "FALLO en top-5"
                ),
            })

    outcomes[metodo] = pd.DataFrame(filas)

    display(Markdown(
        f"### {metodo}\n\n**Pregunta:** {pregunta['pregunta']}"
    ))
    print("Empresa y FY de la pregunta:", filtros_base)

    with pd.option_context("display.max_colwidth", None):
        display(outcomes[metodo])

### 6.1 · Denso + limpieza

**Pregunta:** ¿Qué riesgo declara NVIDIA en FY2025 por depender de un número reducido de proveedores de fabricación?

Empresa y FY de la pregunta: {'ticker': 'NVDA', 'fiscal_year': 2025}


,escenario,item,versión,query,rango_ancla,acierto@5,acierto@10,acierto@20,outcome
0,Sin item,None,Actual del apartado 6,risk depending small number manufacturing suppliers,13,False,False,True,FALLO en top-5
1,Sin item,None,Alternativa manual,dependence limited third party manufacturers risks,Fuera del top-20,False,False,False,FALLO en top-5
2,Con item del golden,1A,Actual del apartado 6,risk depending small number manufacturing suppliers,12,False,False,True,FALLO en top-5
3,Con item del golden,1A,Alternativa manual,dependence limited third party manufacturers risks,Fuera del top-20,False,False,False,FALLO en top-5


### 6.2 · BM25 + palabras clave

**Pregunta:** ¿Cómo evolucionó el beneficio por acción básico de NVIDIA entre los ejercicios fiscales 2024 y 2025, y es esa variación comparable entre ambos ejercicios?

Empresa y FY de la pregunta: {'ticker': 'NVDA', 'fiscal_year': 2025}


,escenario,item,versión,query,rango_ancla,acierto@5,acierto@10,acierto@20,outcome
0,Sin item,None,Actual del apartado 6,basic earnings per share year over year comparability,8,False,True,True,FALLO en top-5
1,Sin item,None,Alternativa manual,basic earnings per share prior year comparison adjusted shares,7,False,True,True,FALLO en top-5
2,Con item del golden,8,Actual del apartado 6,basic earnings per share year over year comparability,7,False,True,True,FALLO en top-5
3,Con item del golden,8,Alternativa manual,basic earnings per share prior year comparison adjusted shares,7,False,True,True,FALLO en top-5


### 6.3 · Denso + reformulación

**Pregunta:** ¿Cómo evolucionó el beneficio por acción básico de NVIDIA entre los ejercicios fiscales 2024 y 2025, y es esa variación comparable entre ambos ejercicios?

Empresa y FY de la pregunta: {'ticker': 'NVDA', 'fiscal_year': 2025}


,escenario,item,versión,query,rango_ancla,acierto@5,acierto@10,acierto@20,outcome
0,Sin item,None,Actual del apartado 6,earnings per share comparison share count adjustments,Fuera del top-20,False,False,False,FALLO en top-5
1,Sin item,None,Alternativa manual,comparability of basic earnings per share across years adjustments to number of shares,Fuera del top-20,False,False,False,FALLO en top-5
2,Con item del golden,8,Actual del apartado 6,earnings per share comparison share count adjustments,Fuera del top-20,False,False,False,FALLO en top-5
3,Con item del golden,8,Alternativa manual,comparability of basic earnings per share across years adjustments to number of shares,Fuera del top-20,False,False,False,FALLO en top-5


## 8. Método seleccionado y prueba real con API

Seleccionamos **BM25 + consulta breve de palabras clave en inglés**, filtrando por empresa y ejercicio fiscal, **sin item**. En las pruebas locales recuperó el ancla en **11 de 13 preguntas dentro del top-5 (84,6 %)** y en **13 de 13 dentro del top-10**. Empata en top-5 con la fusión de denso y BM25, mejora su top-10 y evita combinar dos buscadores.

Las consultas de esa comparación estaban preparadas previamente. Ahora haremos una prueba real mediante API: introduciremos una pregunta en español y el agente extraerá empresa/año, formulará las palabras clave en inglés, buscará con BM25 y responderá en español con una cita. Mostraremos la consulta, los filtros, los fragmentos recuperados y la respuesta para revisar el recorrido completo; un solo ejemplo no valida el rendimiento general.

Ejecuta el arranque y activa `EJECUTAR = True` para usar la siguiente celda. La clave se lee del entorno o de `.env`. Se prueba una pregunta, que puede requerir varias llamadas al modelo, sin ejecutar el juez ni la tanda comparativa.


In [3]:
# Demo independiente de las celdas de comparación con agente.
if not globals().get("EJECUTAR", False):
    print("Demo omitida. Activa EJECUTAR=True y vuelve a ejecutar esta celda.")
else:
    import json
    import time
    from langchain.agents import create_agent
    from langchain.agents.structured_output import ToolStrategy
    from langchain.tools import tool
    from agente10k import agente, config, herramientas, retrieval

    ejemplo = "¿Qué explica Microsoft sobre la evolución de sus ingresos en el ejercicio fiscal 2025?"
    pregunta_demo = ejemplo
    busquedas_demo = []
    documentos_demo = {}

    @tool("search_filings")
    def buscar_bm25_demo(query: str, ticker: str, fiscal_year: int) -> str:
        """Busca los 5 mejores pasajes con palabras clave en inglés, empresa y FY. Sin filtro item."""
        for valor, validar in (
            (ticker, herramientas._validar_ticker),
            (fiscal_year, herramientas._validar_fiscal_year),
        ):
            _, error = validar(valor)
            if error:
                return error
        docs = retrieval.buscar_bm25(query, ticker=ticker, fiscal_year=fiscal_year, k=5)
        busquedas_demo.append({
            "consulta_ingles": query, "empresa": ticker, "ejercicio": fiscal_year,
            "k": 5, "chunks": [d["chunk_id"] for d in docs],
        })
        documentos_demo.update({d["chunk_id"]: d["texto"] for d in docs})
        return "\n\n".join(f"[{d['chunk_id']}] {d['texto']}" for d in docs) or "Sin resultados."

    prompt_demo = """Responde en español usando únicamente las herramientas.
Extrae empresa y ejercicio fiscal de la pregunta. No inventes filtros si faltan:
explica qué información necesitas. Para explicaciones, riesgos y evidencia textual,
formula una consulta breve de palabras clave financieras en inglés y llama a search_filings.
La búsqueda usa BM25 y no admite item. No consultes el golden ni uses consultas prefijadas.
Para cifras usa get_xbrl_fact; en comparativas consulta ambos ejercicios.
Cita una frase literal completa del texto recuperado y su chunk_id exacto.
Si no hay evidencia suficiente, indícalo sin inventar una respuesta.
Devuelve la salida estructurada RespuestaFinanciera; fuente debe reflejar las herramientas usadas."""
    demo_bm25 = create_agent(
        model=config.crear_modelo(),
        tools=[buscar_bm25_demo, herramientas.get_xbrl_fact, herramientas.list_available],
        system_prompt=prompt_demo,
        response_format=ToolStrategy(agente.RespuestaSinFrases),
    )
    print("Modelo:", config.MODELO_ID)
    inicio_demo = time.perf_counter()
    try:
        estado_demo = agente._invocar_con_plazo(
            demo_bm25, {"messages": [{"role": "user", "content": pregunta_demo}]},
            {"recursion_limit": 30}, agente.PLAZO_PREGUNTA_S,
        )
        respuesta_demo = agente.RespuestaSinFrases.model_validate(
            estado_demo["structured_response"]
        ).model_dump()
        print("\nRespuesta:\n", json.dumps(respuesta_demo, ensure_ascii=False, indent=2))
        cita = respuesta_demo.get("cita")
        texto = documentos_demo.get(respuesta_demo.get("chunk_id"), "")
        if cita:
            print("Cita literal en el fragmento recuperado:", cita in texto)
        if not busquedas_demo:
            print("No se utilizó BM25: esta respuesta no comprueba el retrieval.")
    finally:
        print("\nBúsquedas BM25 sin item:")
        print(json.dumps(busquedas_demo, ensure_ascii=False, indent=2))
        print(f"Tiempo total: {time.perf_counter() - inicio_demo:.2f} s")


Modelo: openrouter:google/gemini-3.8-flash

Respuesta:
 {
  "respuesta": "En el ejercicio fiscal 2025, los ingresos de Microsoft aumentaron un 15% ($36.600 millones) hasta alcanzar los $281.724 millones, frente a los $245.122 millones del ejercicio fiscal 2024. Este incremento se debió al crecimiento en todos sus segmentos de negocio: Intelligent Cloud creció un 21% impulsado por Azure; Productivity and Business Processes creció un 13% impulsado por Microsoft 365 Commercial cloud; y More Personal Computing avanzó un 7% impulsado por Gaming (por el impacto de Activision Blizzard y Game Pass) y por publicidad en búsquedas y noticias.",
  "cifra": 281724000000.0,
  "unidad": "USD",
  "ticker": "MSFT",
  "ejercicio": 2025,
  "fuente": "ambas",
  "cita": "Revenue increased $36.6 billion or 15% with growth across each of our segments.",
  "chunk_id": "MSFT-2025-7-0007",
  "concept_xbrl": "RevenueFromContractWithCustomerExcludingAssessedTax",
  "ejercicio_base": 2024,
  "cifra_base": 24512200